# 03 — Construção dos Grafos

Construímos dois grafos complementares:
- **Co-autoria**: autores como nós, co-autoria em um paper como aresta
- **Co-conceito**: temas como nós, co-ocorrência no mesmo paper como aresta

In [ ]:
import sys
sys.path.append('..')

import networkx as nx
from pathlib import Path

from src.fetch import carregar_raw
from src.graph import (
    construir_grafo_coautoria,
    construir_grafo_conceito,
    filtrar_grafo,
    maior_componente,
)
from src.metrics import resumo_rede

FIGURES = Path('../reports/figures')
FIGURES.mkdir(parents=True, exist_ok=True)

works = carregar_raw('works_ufms_cs')

## 1. Grafo de Co-autoria

In [ ]:
G_co = construir_grafo_coautoria(works)
G_co_filtrado = filtrar_grafo(G_co, min_degree=2)
G_co_main = maior_componente(G_co_filtrado)

print('=== Grafo de Co-autoria (completo) ===')
print(resumo_rede(G_co))
print()
print('=== Grafo de Co-autoria (maior componente, grau >= 2) ===')
print(resumo_rede(G_co_main))

## 2. Grafo de Co-conceito

In [ ]:
G_ct = construir_grafo_conceito(works, top_n=5)
G_ct_filtrado = filtrar_grafo(G_ct, min_degree=3)

print('=== Grafo de Co-conceito (completo) ===')
print(resumo_rede(G_ct))
print()
print('=== Grafo de Co-conceito (grau >= 3) ===')
print(resumo_rede(G_ct_filtrado))

## 3. Visualização Estática (matplotlib)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Co-autoria
pos_co = nx.spring_layout(G_co_main, seed=42, k=0.6)
nx.draw_networkx(
    G_co_main, pos=pos_co, ax=axes[0],
    node_size=[G_co_main.degree(n) * 15 + 30 for n in G_co_main.nodes()],
    node_color='#01696f', edge_color='#cccccc',
    font_size=5, with_labels=True,
)
axes[0].set_title('Co-autoria', fontsize=13)
axes[0].axis('off')

# Co-conceito
pos_ct = nx.spring_layout(G_ct_filtrado, seed=42, k=0.4)
nx.draw_networkx(
    G_ct_filtrado, pos=pos_ct, ax=axes[1],
    node_size=[G_ct_filtrado.degree(n) * 10 + 20 for n in G_ct_filtrado.nodes()],
    node_color='#4f98a3', edge_color='#cccccc',
    font_size=6, with_labels=True,
)
axes[1].set_title('Co-conceito', fontsize=13)
axes[1].axis('off')

plt.tight_layout()
plt.savefig(FIGURES / 'grafos_estatico.png', dpi=150)
plt.show()

## 4. Visualização Interativa (PyVis)

In [ ]:
from pyvis.network import Network

net = Network(height='700px', width='100%', notebook=True, cdn_resources='in_line')
net.from_nx(G_co_main)
net.show(str(FIGURES / 'grafo_coautoria_interativo.html'))